# 스타벅스 매장 정보 수집하기

### 실습 개요
이 실습에서는 스타벅스 웹사이트의 매장 검색 API를 활용하여 특정 매장의 상세 정보를 수집하는 방법을 학습합니다.  
크롬 개발자 도구를 활용해 스타벅스 매장 검색 서비스의 POST 요청 구조를 분석하고, 매장명, 주소, 전화번호, 위치 좌표, 주차 정보 등 다양한 매장 정보를 체계적으로 수집하는 방법을 배웁니다.

In [6]:
import requests

def get_starbucks_store_info(in_biz_cd, rnd_cod):
    """
    스타벅스 매장 정보를 가져오는 함수
    
    Args:
        in_biz_cd: 매장 비즈니스 코드
        rnd_cod: 랜덤 코드
    
    Returns:
        dict: 매장 정보 (JSON 형태)
    """
    url = "https://www.starbucks.co.kr/store/getStoreView.do"
    
    # 요청 헤더 설정
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
        'Accept': '*/*',
        'Origin': 'https://www.starbucks.co.kr',
        'Referer': 'https://www.starbucks.co.kr/store/store_map.do'
    }
    
    # POST 데이터
    payload = {
        'in_biz_cd': in_biz_cd,
        'rndCod': rnd_cod
    }
    
    try:
        # POST 요청 보내기
        response = requests.post(url, headers=headers, data=payload)
        response.raise_for_status()  # HTTP 에러 체크
        
        # JSON 응답 반환
        return response.json()
    
    except requests.exceptions.RequestException as e:
        print(f"요청 중 오류 발생: {e}")
        return None

def extract_store_info(raw_data):
    """
    API 응답에서 필요한 매장 정보만 추출
    
    Args:
        raw_data: API 응답 원본 데이터
    
    Returns:
        dict: 정제된 매장 정보
    """
    if not raw_data or 'view' not in raw_data or not raw_data['view']:
        return None
    
    store = raw_data['view'][0]
    
    # 필요한 정보만 추출
    extracted_info = {
        '매장명': store.get('s_name'),
        '매장코드': store.get('s_biz_code'),
        '전화번호': store.get('tel'),
        '팩스': store.get('fax'),
        '배달전화': store.get('dlvry_call_cntr_phno'),
        '주소': {
            '지번주소': store.get('addr'),
            '도로명주소': store.get('doro_address'),
            '시도': store.get('sido_name'),
            '구군': store.get('gugun_name')
        },
        '위치': {
            '위도': store.get('lat'),
            '경도': store.get('lot')
        },
        '주차정보': store.get('park_info'),
        '오시는길': store.get('map_desc'),
        '매장안내': store.get('notice'),
        '이미지': {
            '대표이미지': store.get('defaultimage'),
            '추가이미지': store.get('etcimage', '').split(',') if store.get('etcimage') else []
        },
        '사이렌오더': store.get('my_siren_order_store_yn'),
        '휠체어': store.get('whcroad_yn')
    }
    
    return extracted_info

In [8]:
from pprint import pprint

# 스타벅스의 모든 매장 정보를 수집하고 싶다.
# in_biz_cd 숫자값을 1 ~ 9999 이렇게 반복문을 사용하면되겠다.
in_biz_cd = "6"
rnd_cod = "ACIWYPGRRB"

# 1. 원본 데이터 가져오기
raw_data = get_starbucks_store_info(in_biz_cd, rnd_cod)

if raw_data:
    # 2. 필요한 정보만 추출
    store_info = extract_store_info(raw_data)
    
    # 3. 보기 좋게 출력
    pprint(store_info)

else:
    print("매장 정보를 가져오는데 실패했습니다.")

{'매장명': '둔산은하수',
 '매장안내': '둔산동에서 가장 쾌적하고 친절한 스타벅스 둔산은하수점으로 여러분을 초대합니다. ',
 '매장코드': '9504',
 '배달전화': '1551-3232',
 '사이렌오더': 'N',
 '오시는길': '갤러리아백화점 타임월드점 주차동 옆 ',
 '위치': {'경도': '127.3777879', '위도': '36.3529477'},
 '이미지': {'대표이미지': '/upload/store/2022/03/[9504]_20220327035816_fk430.jpg',
         '추가이미지': ['/upload/store/2023/02/[9504]_20230213081441_izuwu.jpg',
                   '/upload/store/2023/02/[9504]_20230213075715_hnh2s.jpg',
                   '/upload/store/2023/02/[9504]_20230213080943_b1weq.jpg',
                   '/upload/store/2023/02/[9504]_20230213081134_apkeb.jpg',
                   '/upload/store/2023/02/[9504]_20230213081248_9q19h.jpg']},
 '전화번호': '1522-3232',
 '주소': {'구군': '서구',
        '도로명주소': '대전광역시 서구 둔산로31번길 28, 금정빌딩 1층 (둔산동)',
        '시도': '대전',
        '지번주소': '대전광역시 서구 둔산동 1010번지 금정빌딩 1층'},
 '주차정보': '1.주차불가능',
 '팩스': '042-487-8307',
 '휠체어': 'WHCROAD'}
